In [ ]:
# !unzip -q /content/sparse-attention-jax.zip
! git clone https://github.com/sp4s-s/sparse-jax-atn
%cd /content/sparse-jax-atn/
!pip install . -q
!pip install tensorboardX tensorboard -qUU

Cloning into 'sparse-jax-atn'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (61/61), done.
remote: Total 63 (delta 1), reused 63 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 107.02 KiB | 10.70 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/sparse-jax-atn
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.1/155.1 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 101.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.19.0 requires tensorboard~=2.19.0, but you have tensorboard 2.20.0 which is incompatible.


In [ ]:
!cat /sys/kernel/mm/transparent_hugepage/enabled
!mount | grep sysfs
!sudo mount -o remount,rw /sys
!echo always | sudo tee /sys/kernel/mm/transparent_hugepage/enabled
!cat /sys/kernel/mm/transparent_hugepage/enabled

always [madvise] never
sysfs on /sys type sysfs (ro,nosuid,nodev,noexec,relatime)
always
[always] madvise never


In [ ]:
import jax
from train import train
from benchmarks.roofline import roofline_analysis
from benchmarks.mega_stress import run_mega_stress
import sparse_attention

print(jax.devices())
print(jax.default_backend())
print(f"JAX version: {jax.__version__}")
print(f"Package version: {sparse_attention.__version__}")

[TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
tpu
JAX version: 0.7.2
Package version: 1.0.0


In [ ]:
from train import train, run_inference
from benchmarks.roofline import roofline_analysis
from benchmarks.mega_stress import run_mega_stress

# 1. Train the model and catch the returned state!
state = train(attention_type="dense", max_steps=1000, log_dir="tensorboard_logs")

# 2. Run inference using the trained state!
generated_text = run_inference(
    state,
    prompt="[TONY]: Friday, run a diagnostic on my flight stabilizers and prep the arc reactor.",
    max_new_tokens=60
)

# 3. Proceed with your benchmarks
roofline_analysis(
    seq_lengths=[256, 512, 1024, 2048],
    batch_size=1,
    output_dir="results",
)

run_mega_stress(output_dir="results", target_ram_gb=6.0)

In [ ]:
train(attention_type="dense", max_steps=1000, log_dir="tensorboard_logs")
roofline_analysis(
    seq_lengths=[256, 512, 1024, 2048],
    batch_size=1,
    output_dir="results",
)
run_mega_stress(output_dir="results", target_ram_gb=6.0)

training dense | backend=tpu | B=4 N=1024 steps=1000
Log: tensorboard_logs
Initializing model...
Trainable Parameters: 29,465,681

Writing pre-training sample to tensorboard_logs/run_dense_1773078196/generation_before_training.txt
Pre-training sample saved.
Baseline metrics | val_loss=11.2709 ppl=78503.75 acc=0.0000 rep4=0.0074


Training:   0%|          | 0/1000 [00:00<?, ?it/s]

Compiling training step...


Training:   0%|          | 1/1000 [00:33<9:10:30, 33.06s/it, loss=11.2787, tok/s=222434]

Done compiling.



Training: 100%|██████████| 1000/1000 [00:53<00:00, 18.73it/s, loss=0.0025, tok/s=225199]


Writing post-training sample to tensorboard_logs/run_dense_1773078196/generation_after_training.txt
Post-training sample saved.

Final: 18.3ms (224088 tok/s)
Post-training metrics | val_loss=0.0011 ppl=1.00 acc=0.9993 rep4=0.0000
Improvement | delta_loss=11.2698 delta_ppl=78502.75 delta_acc=0.9993 delta_rep4=0.0074
Exporting plots...

GENERATING TRAINING VISUALIZATIONS
  📊 results/train_dense_1773078352/loss_curve.png
  🌐 results/train_dense_1773078352/loss_curve_plotly.html
  📊 results/train_dense_1773078352/grad_norm.png
  📊 results/train_dense_1773078352/lr_schedule.png
  📊 results/train_dense_1773078352/step_throughput.png
  📊 results/train_dense_1773078352/training_dashboard.png
  🌐 results/train_dense_1773078352/training_dashboard_plotly.html
  ✅ Training visuals → results/train_dense_1773078352
Results saved to: results/train_dense_1773078352

ROOFLINE MODEL ANALYSIS
Hardware: TPU v5e-1
Peak compute: 197 TFLOPs (bf16)
Peak bandwidth: 820 GB/s
Ridge point: 240.2 FLOPs/byte

  N=2

{'host_ram': {'status': 'PASS',
  'target_gb': 6.0,
  'achieved_gb': 6.0,
  'attention_under_pressure_ms': 1278.3,
  'available_after_gb': 31.96,
  'chunk_mb': 64,
  'safety_headroom_gb': 4.5},
 'ceiling': {'sparse': [{'config': [1, 1024, 8, 64],
    'est_gb': 0.0,
    'status': 'PASS',
    'latency_ms': 798.02,
    'tok_s': 1283.17},
   {'config': [2, 2048, 8, 64],
    'est_gb': 0.01,
    'status': 'PASS',
    'latency_ms': 1206.34,
    'tok_s': 3395.38},
   {'config': [4, 4096, 8, 64],
    'est_gb': 0.05,
    'status': 'PASS',
    'latency_ms': 1691.27,
    'tok_s': 9687.39},
   {'config': [8, 4096, 8, 64],
    'est_gb': 0.1,
    'status': 'PASS',
    'latency_ms': 1509.02,
    'tok_s': 21714.77},
   {'config': [8, 8192, 8, 64],
    'est_gb': 0.2,
    'status': 'PASS',
    'latency_ms': 1849.25,
    'tok_s': 35439.18},
   {'config': [8, 8192, 16, 64],
    'est_gb': 0.4,
    'status': 'PASS',
    'latency_ms': 2016.62,
    'tok_s': 32497.91},
   {'config': [16, 8192, 16, 64],
    'est

In [ ]:
# Agressive run ram size Fails ≥ 6 gigs || ≥ 16 [ batch size]
# Obs max ram ig 7.34G
roofline_analysis(
    seq_lengths=[256, 512, 1024, 2048, 4096],
    batch_size=8,
    output_dir="results_x2",
)
run_mega_stress(output_dir="results_x2", target_ram_gb=6.0)


ROOFLINE MODEL ANALYSIS
Hardware: TPU v5e-1
Peak compute: 197 TFLOPs (bf16)
Peak bandwidth: 820 GB/s
Ridge point: 240.2 FLOPs/byte

  N=256:
    [ dense] AI=44.3 FLOPs/B, 0.2868 TFLOPs, MFU=100.0%, HFU=0.1%, BW=6 GB/s (1%), Tok/s=1052959, P99=2.07ms [MEM-BOUND]
    [sparse] AI=99.8 FLOPs/B, 0.0008 TFLOPs, MFU=100.0%, HFU=0.0%, BW=0 GB/s (0%), Tok/s=4011, P99=1016.13ms [MEM-BOUND]

  N=512:
    [ dense] AI=53.2 FLOPs/B, 0.8987 TFLOPs, MFU=100.0%, HFU=0.5%, BW=17 GB/s (2%), Tok/s=1649606, P99=2.63ms [MEM-BOUND]
    [sparse] AI=166.2 FLOPs/B, 0.0030 TFLOPs, MFU=100.0%, HFU=0.0%, BW=0 GB/s (0%), Tok/s=8676, P99=506.29ms [MEM-BOUND]

  N=1024:
    [ dense] AI=59.1 FLOPs/B, 2.3780 TFLOPs, MFU=100.0%, HFU=1.2%, BW=40 GB/s (5%), Tok/s=2182595, P99=3.85ms [MEM-BOUND]
    [sparse] AI=249.4 FLOPs/B, 0.0087 TFLOPs, MFU=100.0%, HFU=0.0%, BW=0 GB/s (0%), Tok/s=17024, P99=495.93ms [COMPUTE-BOUND]

  N=2048:
    [ dense] AI=62.6 FLOPs/B, 2.9770 TFLOPs, MFU=100.0%, HFU=1.5%, BW=48 GB/s (6%), Tok/s=136

{'host_ram': {'status': 'PASS',
  'target_gb': 6.0,
  'achieved_gb': 6.0,
  'attention_under_pressure_ms': 516.32,
  'available_after_gb': 28.25,
  'chunk_mb': 64,
  'safety_headroom_gb': 4.5},
 'ceiling': {'sparse': [{'config': [1, 1024, 8, 64],
    'est_gb': 0.0,
    'status': 'PASS',
    'latency_ms': 411.27,
    'tok_s': 2489.83},
   {'config': [2, 2048, 8, 64],
    'est_gb': 0.01,
    'status': 'PASS',
    'latency_ms': 518.53,
    'tok_s': 7899.27},
   {'config': [4, 4096, 8, 64],
    'est_gb': 0.05,
    'status': 'PASS',
    'latency_ms': 523.93,
    'tok_s': 31271.55},
   {'config': [8, 4096, 8, 64],
    'est_gb': 0.1,
    'status': 'PASS',
    'latency_ms': 594.45,
    'tok_s': 55122.99},
   {'config': [8, 8192, 8, 64],
    'est_gb': 0.2,
    'status': 'PASS',
    'latency_ms': 634.74,
    'tok_s': 103249.19},
   {'config': [8, 8192, 16, 64],
    'est_gb': 0.4,
    'status': 'PASS',
    'latency_ms': 876.39,
    'tok_s': 74779.12},
   {'config': [16, 8192, 16, 64],
    'est_gb